# Unemployment Analysis in India

I used this notebook to understand how unemployment changed across Indian regions from 2019 to 2020. The main things I wanted to check were:

- how unemployment moved month by month
- which regions had higher unemployment on average
- whether rural and urban areas looked different
- how much the early Covid period changed the pattern

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 50)

## Loading the data

I am using the cleaned file when it is available. If not, the notebook falls back to the original CSV and applies the same basic cleaning steps.

In [ ]:
BASE_DIR = Path.cwd()
CLEANED_DATA_PATH = BASE_DIR / "data" / "cleaned_unemployment_india.csv"
RAW_DATA_PATH = BASE_DIR / "data" / "Unemployment in India.csv"

data_path = CLEANED_DATA_PATH if CLEANED_DATA_PATH.exists() else RAW_DATA_PATH
df = pd.read_csv(data_path)

print("Data file used:", data_path.name)
df.head()

## Cleaning notes

The raw file has a few extra spaces in column names and text values, so I standardised those first. I also recreated the date-related columns so the rest of the notebook works even if it is run from the raw dataset.

In [ ]:
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("%", "percent", regex=False)
)

df["date"] = pd.to_datetime(df["date"], errors="coerce", dayfirst=True)

for column in ["frequency", "region", "area"]:
    if column in df.columns:
        df[column] = df[column].astype(str).str.strip()

df = df.drop_duplicates().sort_values(["date", "region", "area"]).reset_index(drop=True)
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_number"] = df["date"].dt.month
df["period"] = df["date"].dt.to_period("M").astype(str)

df.head()

In [ ]:
# Quick quality check before doing any analysis
pd.DataFrame(
    {
        "missing_values": df.isna().sum(),
        "unique_values": df.nunique(),
    }
)

## First look at the dataset

In [ ]:
overview = pd.Series(
    {
        "rows": df.shape[0],
        "columns": df.shape[1],
        "start_date": df["date"].min(),
        "end_date": df["date"].max(),
        "regions": df["region"].nunique(),
        "areas": ", ".join(sorted(df["area"].dropna().unique())),
    }
)

overview.to_frame("value")

In [ ]:
numeric_cols = [
    "estimated_unemployment_rate_percent",
    "estimated_employed",
    "estimated_labour_participation_rate_percent",
]

df[numeric_cols].describe().T

The dataset is small enough to inspect directly, but the trend chart is more useful for spotting the 2020 jump.

## Month-by-month unemployment

In [ ]:
monthly = (
    df.groupby("date", as_index=False)
    .agg(
        unemployment_rate=("estimated_unemployment_rate_percent", "mean"),
        labour_participation=("estimated_labour_participation_rate_percent", "mean"),
        employed=("estimated_employed", "sum"),
    )
    .sort_values("date")
)

plt.figure(figsize=(11, 5))
sns.lineplot(data=monthly, x="date", y="unemployment_rate", marker="o", linewidth=2.5)
plt.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2020-06-30"), color="#d45d4c", alpha=0.15, label="Covid period")
plt.title("Average unemployment rate over time")
plt.xlabel("Date")
plt.ylabel("Unemployment rate (%)")
plt.legend()
plt.tight_layout()
plt.show()

The shaded part marks March to June 2020. This makes the lockdown-period change much easier to see.

## Regional comparison

In [ ]:
region_avg = (
    df.groupby("region", as_index=False)["estimated_unemployment_rate_percent"]
    .mean()
    .sort_values("estimated_unemployment_rate_percent", ascending=False)
)

plt.figure(figsize=(10, 8))
sns.barplot(data=region_avg, x="estimated_unemployment_rate_percent", y="region")
plt.title("Average unemployment rate by region")
plt.xlabel("Unemployment rate (%)")
plt.ylabel("Region")
plt.tight_layout()
plt.show()

region_avg.head(10)

Tripura and Haryana stand out clearly in the average regional numbers. I kept the full ranking visible because the middle regions are also useful for comparison.

## Rural and urban split

In [ ]:
area_summary = (
    df.groupby("area")
    .agg(
        avg_unemployment=("estimated_unemployment_rate_percent", "mean"),
        median_unemployment=("estimated_unemployment_rate_percent", "median"),
        avg_labour_participation=("estimated_labour_participation_rate_percent", "mean"),
        avg_employed=("estimated_employed", "mean"),
    )
    .round(2)
)

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="area", y="estimated_unemployment_rate_percent")
plt.title("Rural vs urban unemployment distribution")
plt.xlabel("Area")
plt.ylabel("Unemployment rate (%)")
plt.tight_layout()
plt.show()

area_summary

Urban unemployment is higher on average in this dataset, though the spread matters too. The boxplot helps show that the difference is not just one or two extreme values.

## Relationship between the numeric columns

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="crest", fmt=".2f", linewidths=1)
plt.title("Correlation between key indicators")
plt.tight_layout()
plt.show()

## Highest and lowest records

I checked the exact rows for the maximum and minimum unemployment rates because averages can hide the most extreme observations.

In [ ]:
highest_row = df.loc[df["estimated_unemployment_rate_percent"].idxmax()]
lowest_row = df.loc[df["estimated_unemployment_rate_percent"].idxmin()]

pd.DataFrame([highest_row, lowest_row], index=["highest", "lowest"])[
    ["region", "area", "date", "estimated_unemployment_rate_percent", "estimated_employed", "estimated_labour_participation_rate_percent"]
]

## Covid-period comparison

For a simple comparison, I treated March 2020 to June 2020 as the Covid-period window and compared it with all earlier observations.

In [ ]:
covid_start = pd.Timestamp("2020-03-01")
covid_end = pd.Timestamp("2020-06-30")

before_covid = df[df["date"] < covid_start]["estimated_unemployment_rate_percent"]
covid_period = df[(df["date"] >= covid_start) & (df["date"] <= covid_end)]["estimated_unemployment_rate_percent"]

t_stat, p_value = stats.ttest_ind(before_covid, covid_period, equal_var=False, nan_policy="omit")

pd.Series(
    {
        "before_covid_average": before_covid.mean(),
        "covid_period_average": covid_period.mean(),
        "difference": covid_period.mean() - before_covid.mean(),
        "welch_t_statistic": t_stat,
        "p_value": p_value,
    }
).round(4)

## Final notes

In [ ]:
print("Final notes")
print(f"- Dataset covers {df['region'].nunique()} regions from {df['date'].min().date()} to {df['date'].max().date()}.")
print(f"- Overall average unemployment rate: {df['estimated_unemployment_rate_percent'].mean():.2f}%")
print(f"- Highest average unemployment region: {region_avg.iloc[0]['region']} ({region_avg.iloc[0]['estimated_unemployment_rate_percent']:.2f}%)")
print(f"- Highest single observation: {highest_row['region']} / {highest_row['area']} on {highest_row['date'].date()} ({highest_row['estimated_unemployment_rate_percent']:.2f}%)")
print(f"- Covid-period average was {covid_period.mean() - before_covid.mean():.2f} percentage points higher than the pre-Covid average.")